In [1]:
# ============================================================
# STRATEGY C ONLY — TWO-STAGE RULE + LINEAR MODEL (RAW dev)
# Variant requested:
# - Drops DEV rows with missing timestamp (keeps only valid-timestamp rows)
# - BUT: does NOT use timestamp-derived features (NO year/month/dow)
# - Numeric features: n_tokens, title_len, article_len, title_ratio
# - Model: OHE(source) + word tfidf + char tfidf + scaled numeric -> LogisticRegression
# - Two-stage: mine pure token rules on TRAIN ONLY, override predictions on TEST when matched
# - Runs 1 fold (fast sanity)
# ============================================================

import pandas as pd
import numpy as np

from collections import Counter, defaultdict
from sklearn.model_selection import StratifiedKFold
from sklearn.compose import ColumnTransformer
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import f1_score, classification_report, confusion_matrix

# =========================
# CONFIG — STRATEGY C
# =========================
DATA_PATH = "../data/raw/development.csv"

N_SPLITS = 5
RANDOM_STATE = 42
USE_ONLY_FIRST_FOLD = True

# Strategy C params
MIN_RULE_SUPPORT = 30
MIN_RULE_PURITY  = 0.95
RULE_PRIORITY    = "best_purity_then_freq"

# Model params
C_VALUE = 1.5

# TF-IDF sizes
WORD_MAX_FEATURES = 250_000
CHAR_MAX_FEATURES = 300_000

# Print controls
PRINT_FULL_REPORT = True

# =========================
# LOAD + TIMESTAMP DROP (DEV ONLY)
# =========================
df = pd.read_csv(DATA_PATH)

print("Samples after timestamp drop:", len(df))

# =========================
# BASIC FIXES
# =========================
df["article"] = df["article"].fillna("").astype(str)
df["title"]   = df["title"].fillna("").astype(str)
df["source"]  = df["source"].fillna("").astype(str)

# =========================
# TEXT
# =========================
df["text"] = (df["title"] + " " + df["article"]).str.lower()

# =========================
# NUMERIC FEATURES (NO TIME DERIVATIVES)
# =========================
df["n_tokens"]    = df["article"].str.split().str.len()
df["title_len"]   = df["title"].str.len()
df["article_len"] = df["article"].str.len()
df["title_ratio"] = df["title_len"] / (df["article_len"] + 1)

# Ensure numeric are clean (defensive)
NUM_COLS = ["n_tokens", "title_len", "article_len", "title_ratio"]
df[NUM_COLS] = df[NUM_COLS].replace([np.inf, -np.inf], np.nan).fillna(0)

# =========================
# X / y
# =========================
FEATURES = ["source", "text"] + NUM_COLS
X = df[FEATURES].copy()
y = df["label"].astype(int).copy()

# =========================
# RULE TOKENIZATION
# =========================
def tokenize_for_rules(text: str):
	# keep it simple and FAST; preserves html-ish tokens like '/><img'
	return text.split()

# =========================
# RULE MINING (TRAIN ONLY)
# =========================
def mine_pure_rules(texts, labels, min_support, min_purity):
	counts = defaultdict(lambda: Counter())

	for txt, yy in zip(texts, labels):
		for tok in set(tokenize_for_rules(txt)):
			counts[tok][int(yy)] += 1

	rule_token_to_class = {}
	rule_meta = {}

	for tok, c in counts.items():
		total = sum(c.values())
		if total < min_support:
			continue

		best_class, best_freq = c.most_common(1)[0]
		purity = best_freq / total

		if purity >= min_purity:
			rule_token_to_class[tok] = int(best_class)
			rule_meta[tok] = (float(purity), int(total))

	return rule_token_to_class, rule_meta

# =========================
# APPLY RULES
# =========================
def apply_rules(texts, rule_token_to_class, rule_meta, priority):
	rule_pred = np.full(len(texts), -1, dtype=int)
	matched_token = [None] * len(texts)

	for i, txt in enumerate(texts):
		toks = set(tokenize_for_rules(txt))
		hits = [t for t in toks if t in rule_token_to_class]
		if not hits:
			continue

		if priority == "best_purity_then_freq":
			hits.sort(key=lambda t: (rule_meta[t][0], rule_meta[t][1]), reverse=True)
		elif priority == "freq_then_purity":
			hits.sort(key=lambda t: (rule_meta[t][1], rule_meta[t][0]), reverse=True)
		else:
			hits.sort(key=lambda t: (rule_meta[t][0], rule_meta[t][1]), reverse=True)

		best = hits[0]
		rule_pred[i] = int(rule_token_to_class[best])
		matched_token[i] = best

	return rule_pred, matched_token

# =========================
# MODEL
# =========================
def make_model(C):
	pre = ColumnTransformer(
		transformers=[
			("src", OneHotEncoder(handle_unknown="ignore"), ["source"]),
			("w_tfidf", TfidfVectorizer(
				analyzer="word",
				ngram_range=(1, 2),
				min_df=3,
				max_df=0.9,
				sublinear_tf=True,
				max_features=WORD_MAX_FEATURES
			), "text"),
			("c_tfidf", TfidfVectorizer(
				analyzer="char_wb",
				ngram_range=(3, 5),
				min_df=3,
				max_df=0.9,
				sublinear_tf=True,
				max_features=CHAR_MAX_FEATURES
			), "text"),
			("num", StandardScaler(), NUM_COLS),
		],
		remainder="drop",
		n_jobs=-1
	)

	clf = LogisticRegression(
		C=C,
		class_weight="balanced",
		max_iter=2000,
		n_jobs=-1
	)

	return Pipeline([
		("pre", pre),
		("clf", clf),
	])

# =========================
# RUN (1 FOLD)
# =========================
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=RANDOM_STATE)

for fold_id, (tr, te) in enumerate(skf.split(X, y), start=1):
	print(f"\n===== FOLD {fold_id} =====")

	X_tr, y_tr = X.iloc[tr].copy(), y.iloc[tr].copy()
	X_te, y_te = X.iloc[te].copy(), y.iloc[te].copy()

	# ---- Mine rules on TRAIN ONLY
	rule_token_to_class, rule_meta = mine_pure_rules(
		X_tr["text"], y_tr,
		min_support=MIN_RULE_SUPPORT,
		min_purity=MIN_RULE_PURITY
	)
	print("Mined rules:", len(rule_token_to_class))

	# ---- Train model
	model = make_model(C=C_VALUE)
	model.fit(X_tr, y_tr)

	# ---- Base predictions
	model_pred = model.predict(X_te)

	# ---- Apply rules
	rule_pred, matched_token = apply_rules(
		X_te["text"],
		rule_token_to_class,
		rule_meta,
		priority=RULE_PRIORITY
	)

	final_pred = model_pred.copy()
	mask = (rule_pred != -1)
	final_pred[mask] = rule_pred[mask]

	# ---- Metrics
	macro = f1_score(y_te, final_pred, average="macro")
	print(f"Macro F1: {macro:.6f}")
	print(f"Rule coverage: {mask.mean():.3f} ({mask.sum()} / {len(mask)})")

	if mask.any():
		rule_only = f1_score(y_te[mask], final_pred[mask], average="macro")
		print(f"Rule-only Macro F1 (matched subset): {rule_only:.6f}")

	counter = Counter([t for t in matched_token if t is not None])
	print("Top matched rule tokens:", counter.most_common(20))

	if PRINT_FULL_REPORT:
		print("\nConfusion Matrix:\n", confusion_matrix(y_te, final_pred))
		print("\nReport:\n", classification_report(y_te, final_pred, digits=3))

	if USE_ONLY_FIRST_FOLD:
		break

Samples after timestamp drop: 79997

===== FOLD 1 =====
Mined rules: 92
Macro F1: 0.712577
Rule coverage: 0.055 (885 / 16000)
Rule-only Macro F1 (matched subset): 0.796058
Top matched rule tokens: [('/><img', 109), ('details.\\', 55), ('(canadian', 36), ('health)', 34), ('alt="democratic', 32), ('alt="republican', 28), ("href='http://www.newsisfree.com/sources/info/2315/'>cnn</a>", 25), ('mets', 25), ('night.</p><br', 22), ('<p>the', 22), ('(pc', 22), ('sep.', 22), ('(healthday', 19), ('jun.', 18), ('(infoworld)\\', 17), ('sport:', 17), ('...<br/><a', 16), ('alt=""></a>\\', 14), ('steelers', 14), ('three-run', 14)]

Confusion Matrix:
 [[3304  158  106  298   42  710   90]
 [  78 1722  106   72   18   74   47]
 [  67  130 1842   74    8   52   59]
 [ 190  111  109 1132  127  257   70]
 [  15   11    2   66 1565   50    6]
 [ 500  134   61  335  125 1371   85]
 [  32   16    9   29    5   35  495]]

Report:
               precision    recall  f1-score   support

           0      0.789  